In [ ]:
!unzip /content/dev_phase.zip

Archive:  /content/dev_phase.zip
   creating: subtask1/
   creating: subtask1/dev/
  inflating: subtask1/dev/nep.csv    
  inflating: subtask1/dev/ita.csv    
  inflating: subtask1/dev/pol.csv    
  inflating: subtask1/dev/rus.csv    
  inflating: subtask1/dev/tel.csv    
  inflating: subtask1/dev/hin.csv    
  inflating: subtask1/dev/hau.csv    
  inflating: subtask1/dev/pan.csv    
  inflating: subtask1/dev/ori.csv    
  inflating: subtask1/dev/spa.csv    
  inflating: subtask1/dev/deu.csv    
  inflating: subtask1/dev/fas.csv    
  inflating: subtask1/dev/arb.csv    
  inflating: subtask1/dev/ben.csv    
  inflating: subtask1/dev/amh.csv    
  inflating: subtask1/dev/khm.csv    
  inflating: subtask1/dev/tur.csv    
  inflating: subtask1/dev/zho.csv    
  inflating: subtask1/dev/eng.csv    
  inflating: subtask1/dev/swa.csv    
  inflating: subtask1/dev/urd.csv    
  inflating: subtask1/dev/mya.csv    
   creating: subtask1/train/
  inflating: subtask1/train/nep.csv  
  inflating: s

In [ ]:
# @title
import os
import re
import glob
import math
import shutil
import random
from dataclasses import dataclass
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, hamming_loss

In [ ]:
# @title
import wandb

# Disable wandb logging for this script
wandb.init(mode="disabled")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


In [ ]:
# @title
LABEL_KEYS = [
    "stereotype",
    "vilification",
    "dehumanization",
    "extreme_language",
    "lack_of_empathy",
    "invalidation"
]

SEED = 42
MAX_LEN = 256

# Focal loss hyperparams (reasonable defaults; tune gamma/alpha lightly)
FOCAL_ALPHA = 0.25
FOCAL_GAMMA = 2.0

# Threshold tuning grid (denser grid tends to help Macro-F1)
THRESH_GRID = np.linspace(0.05, 0.95, 37)

# Language balancing strength (increase slightly if low-resource langs suffer)
LANG_BALANCE_POWER = 1.0  # 1.0 = inverse freq; 0.5 = softer

# Label-aware example weighting strength
LABEL_AWARE_WEIGHT = 1.0  # 0.0 disables label-aware weighting

# Optional LoRA
USE_LORA_IF_AVAILABLE = True

# ==================== Utilities ====================
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

def normalize_columns(df):
    """Cleans up column names for consistency."""
    df.columns = df.columns.str.replace("\n", " ").str.strip()
    df.columns = df.columns.str.replace(r"\s+", " ", regex=True)
    df.columns = df.columns.str.lower().str.replace(" ", "_")
    return df

def sigmoid_np(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))

# ==================== Load and Prepare Data ====================
# Your existing paths + an additional fallback for uploaded CSVs (/mnt/data)
possible_paths = [
    '/content/dev_aug/subtask3/train/',
    '/content/subtask3/train/',
    '/content/train/',
    './train/',
    './subtask3/train/',
    '/mnt/data/',  # fallback for uploaded csvs
]

base_path = None
for path in possible_paths:
    if os.path.exists(path):
        files = glob.glob(os.path.join(path, "*.csv"))
        # Heuristic: training folder should have multiple language csvs
        if len(files) >= 2:
            base_path = path
            print(f"✓ Found training data in: {base_path}")
            print(f"  Files found: {len(files)}")
            break

if base_path is None:
    print("❌ ERROR: Could not find training CSV files.")
    print("Please check your data location and update base_path.")
    print("\nCurrent directory contents (/content):")
    try:
        print(os.listdir('/content/'))
    except Exception:
        print("  (No /content directory in this runtime.)")
    print("\nCurrent directory contents (/mnt/data):")
    try:
        print(os.listdir('/mnt/data/'))
    except Exception:
        print("  (No /mnt/data directory in this runtime.)")
    raise ValueError("Training data not found. Please upload your data or update the path.")

all_files = glob.glob(os.path.join(base_path, "*.csv"))

df_list = []
for f in all_files:
    lang_code = os.path.basename(f).split('.')[0]
    temp_df = pd.read_csv(f)
    temp_df['lang'] = lang_code
    df_list.append(temp_df)
    print(f"  Loaded {lang_code}: {len(temp_df)} rows")

df_all = pd.concat(df_list, ignore_index=True)
df_all = normalize_columns(df_all)

# Ensure required columns exist
if "text" not in df_all.columns:
    raise ValueError("Expected a 'text' column in training CSVs, but none was found.")

# Fill missing label columns with 0; coerce to int
for k in LABEL_KEYS:
    if k in df_all.columns:
        df_all[k] = df_all[k].fillna(0).astype(int)
    else:
        df_all[k] = 0

# ==================== Train/Val Split ====================
# Note: sklearn stratify does not support multi-label matrix directly.
# Keep your try/except, but remove stratify on multioutput to avoid silent issues.
train, val = train_test_split(
    df_all,
    test_size=0.15,
    random_state=SEED
)

train = train.reset_index(drop=True)
val = val.reset_index(drop=True)

print(f"\nTraining set size: {len(train)}")
print(f"Validation set size: {len(val)}")
print(f"Training languages: {sorted(train['lang'].unique().tolist())}")

✓ Found training data in: /content/subtask3/train/
  Files found: 18
  Loaded khm: 6640 rows
  Loaded ori: 2368 rows
  Loaded hau: 3651 rows
  Loaded spa: 3305 rows
  Loaded deu: 3180 rows
  Loaded ben: 3333 rows
  Loaded hin: 2744 rows
  Loaded fas: 3295 rows
  Loaded tur: 2364 rows
  Loaded urd: 3563 rows
  Loaded pan: 1700 rows
  Loaded arb: 3380 rows
  Loaded nep: 2005 rows
  Loaded tel: 2366 rows
  Loaded zho: 4280 rows
  Loaded swa: 6991 rows
  Loaded eng: 3222 rows
  Loaded amh: 3332 rows

Training set size: 52461
Validation set size: 9258
Training languages: ['amh', 'arb', 'ben', 'deu', 'eng', 'fas', 'hau', 'hin', 'khm', 'nep', 'ori', 'pan', 'spa', 'swa', 'tel', 'tur', 'urd', 'zho']


In [ ]:
# @title
# ==================== Dataset Class ====================
class PolarizationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = list(texts)
        self.labels = np.array(labels, dtype=np.float32)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding=False,
            max_length=self.max_length,
            return_tensors='pt'
        )
        item = {key: encoding[key].squeeze(0) for key in encoding.keys()}
        item['labels'] = torch.tensor(label, dtype=torch.float32)
        return item

class PredictionDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=256):
        self.texts = list(texts)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding=False,
            max_length=self.max_length,
            return_tensors='pt'
        )
        item = {key: encoding[key].squeeze(0) for key in encoding.keys()}
        return item

In [ ]:
# @title
# ==================== Multi-label Class Weights (pos_weight) ====================
def compute_pos_weight_multilabel(df: pd.DataFrame, label_keys, eps: float = 1.0) -> torch.Tensor:
    """
    For BCEWithLogitsLoss multi-label, use pos_weight per label:
      pos_weight = N_neg / N_pos
    eps is additive smoothing to avoid division-by-zero and extreme spikes.
    """
    y = df[label_keys].astype(float)
    n = len(y)
    pos = y.sum(axis=0).values  # [L]
    neg = n - pos

    pos_s = pos + eps
    neg_s = neg + eps
    pw = (neg_s / pos_s).astype(np.float32)

    pw_t = torch.tensor(pw, dtype=torch.float32)

    print("\nPositive class pos_weight (N_neg / N_pos) per label (smoothed):")
    for k, w, p in zip(label_keys, pw_t.tolist(), pos.tolist()):
        print(f"  {k:18s} pos={int(p):5d}  pos_weight={w:.3f}")

    return pw_t

pos_weights = compute_pos_weight_multilabel(train, LABEL_KEYS, eps=1.0)

# ==================== Focal Loss (Correct Multi-label Variant) ====================
def focal_loss_with_logits(
    logits: torch.Tensor,
    targets: torch.Tensor,
    pos_weight: torch.Tensor,
    gamma: float = 2.0,
    alpha: float = 0.25,
    reduction: str = "mean",
):
    """
    Multi-label focal loss built on BCEWithLogitsLoss (with pos_weight).
    logits:  [B, L]
    targets: [B, L] in {0,1}
    pos_weight: [L]
    """
    # BCE per element, already incorporating per-label pos_weight
    bce = nn.functional.binary_cross_entropy_with_logits(
        logits, targets, reduction="none", pos_weight=pos_weight.to(logits.device)
    )  # [B, L]

    # Probabilities
    p = torch.sigmoid(logits)

    # pt = p if y=1 else 1-p
    pt = p * targets + (1.0 - p) * (1.0 - targets)

    # alpha_t = alpha if y=1 else 1-alpha
    alpha_t = alpha * targets + (1.0 - alpha) * (1.0 - targets)

    loss = alpha_t * ((1.0 - pt) ** gamma) * bce  # [B, L]

    if reduction == "mean":
        return loss.mean()
    elif reduction == "sum":
        return loss.sum()
    return loss

# ==================== Language-balanced + label-aware sampler ====================
def build_example_weights(df: pd.DataFrame, label_keys, pos_weight: torch.Tensor) -> np.ndarray:
    """
    Builds per-example weights for WeightedRandomSampler:
      weight_i = inv_lang_freq(lang)^power * (1 + LABEL_AWARE_WEIGHT * mean(pos_weight[label]==1 for labels present))
    """
    # Language inverse frequency
    lang_counts = df["lang"].value_counts().to_dict()
    inv_lang = {l: (1.0 / c) ** LANG_BALANCE_POWER for l, c in lang_counts.items()}

    y = df[label_keys].values.astype(np.float32)  # [N, L]
    pw = pos_weight.detach().cpu().numpy().astype(np.float32)  # [L]

    # label-aware part: mean of pos_weight for positive labels; if no positives -> 0
    pos_mask = y > 0.5
    # sum weights for positive labels per example
    pos_weight_sum = (pos_mask * pw[None, :]).sum(axis=1)
    pos_count = pos_mask.sum(axis=1)
    pos_mean = np.where(pos_count > 0, pos_weight_sum / pos_count, 0.0)

    label_factor = 1.0 + (LABEL_AWARE_WEIGHT * pos_mean)

    # language factor
    lang_factor = np.array([inv_lang[l] for l in df["lang"].tolist()], dtype=np.float32)

    w = (lang_factor * label_factor).astype(np.float32)

    # Safety normalization to avoid extreme weights
    w = np.clip(w, a_min=np.percentile(w, 1), a_max=np.percentile(w, 99))
    w = w / (w.mean() + 1e-8)

    print("\nSampler diagnostics:")
    print(f"  Example weights: min={w.min():.3f}, mean={w.mean():.3f}, max={w.max():.3f}")
    print("  Language counts (train):", dict(sorted(lang_counts.items(), key=lambda x: x[0])))

    return w

train_example_weights = build_example_weights(train, LABEL_KEYS, pos_weights)

# ==================== Metrics (monitoring at threshold=0.5) ====================
def compute_metrics_multilabel(eval_pred):
    logits, labels = eval_pred
    probs = sigmoid_np(logits)
    preds = (probs >= 0.5).astype(int)

    labels_int = labels.astype(int)

    per_label_f1 = f1_score(labels_int, preds, average=None, zero_division=0)

    metrics = {
        "f1_macro": f1_score(labels_int, preds, average="macro", zero_division=0),
        "f1_micro": f1_score(labels_int, preds, average="micro", zero_division=0),
        "hamming_loss": hamming_loss(labels_int, preds),
    }
    for k, s in zip(LABEL_KEYS, per_label_f1):
        metrics[f"f1_{k}"] = float(s)
    return metrics

# ==================== Per-label threshold tuning (for Macro-F1) ====================
def tune_thresholds_per_label(val_logits: np.ndarray, val_true: np.ndarray, label_keys, grid: np.ndarray):
    probs = sigmoid_np(val_logits)
    thresholds = np.zeros(probs.shape[1], dtype=float)
    best_f1s = np.zeros(probs.shape[1], dtype=float)

    for j in range(probs.shape[1]):
        yj = val_true[:, j].astype(int)
        pj = probs[:, j]

        best_f1 = -1.0
        best_t = 0.5

        for t in grid:
            pred = (pj >= t).astype(int)
            f1 = f1_score(yj, pred, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t = float(t)

        thresholds[j] = best_t
        best_f1s[j] = best_f1

    print("\nBest per-label thresholds (optimized on validation):")
    for k, t, f in zip(label_keys, thresholds, best_f1s):
        print(f"  {k:18s} thr={t:.2f}  val_f1={f:.4f}")

    return thresholds

def predict_with_thresholds(logits: np.ndarray, thresholds: np.ndarray):
    probs = sigmoid_np(logits)
    preds = (probs >= thresholds[None, :]).astype(int)
    return preds, probs

# ==================== Custom Trainer (Focal loss + weighted sampler) ====================
class FocalLossWeightedSamplerTrainer(Trainer):
    def __init__(self, pos_weight=None, alpha=0.25, gamma=2.0, sample_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight
        self.alpha = alpha
        self.gamma = gamma
        self.sample_weights = sample_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").float()
        outputs = model(**inputs)
        logits = outputs.logits

        loss = focal_loss_with_logits(
            logits=logits,
            targets=labels,
            pos_weight=self.pos_weight,
            gamma=self.gamma,
            alpha=self.alpha,
            reduction="mean",
        )
        return (loss, outputs) if return_outputs else loss

    def get_train_dataloader(self):
        # Use WeightedRandomSampler if provided; otherwise fall back to default Trainer dataloader.
        if self.sample_weights is None:
            return super().get_train_dataloader()

        sampler = WeightedRandomSampler(
            weights=torch.tensor(self.sample_weights, dtype=torch.float),
            num_samples=len(self.sample_weights),
            replacement=True
        )
        return DataLoader(
            self.train_dataset,
            batch_size=self.args.train_batch_size,
            sampler=sampler,
            collate_fn=self.data_collator,
            num_workers=self.args.dataloader_num_workers,
            pin_memory=True,
        )

# ==================== Load Model and Tokenizer ====================
MODEL_NAME = "Sami92/XLM-R-Large-Polarization-Classifier"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_KEYS),
    problem_type="multi_label_classification",
    ignore_mismatched_sizes=True
)

# Gradient checkpointing helps memory and can improve effective batch size stability.
try:
    model.gradient_checkpointing_enable()
    print("\n✓ Enabled gradient checkpointing")
except Exception as e:
    print("\n(i) Gradient checkpointing not enabled:", str(e))

# ==================== Optional LoRA (PEFT) ====================
if USE_LORA_IF_AVAILABLE:
    try:
        from peft import LoraConfig, get_peft_model, TaskType

        lora_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            target_modules=["query", "key", "value"],
        )
        model = get_peft_model(model, lora_config)
        print("\n✓ Using LoRA adapters (PEFT)")
        try:
            model.print_trainable_parameters()
        except Exception:
            pass
    except Exception as e:
        print("\n(i) PEFT/LoRA not available; continuing without LoRA.")
        print("    Reason:", str(e))

# ==================== Create Datasets ====================
train_dataset = PolarizationDataset(
    train['text'].tolist(),
    train[LABEL_KEYS].values.tolist(),
    tokenizer,
    max_length=MAX_LEN
)
val_dataset = PolarizationDataset(
    val['text'].tolist(),
    val[LABEL_KEYS].values.tolist(),
    tokenizer,
    max_length=MAX_LEN
)

# ==================== Training Arguments ====================
training_args = TrainingArguments(
    output_dir=f"./checkpoints_xlmr_large",
    num_train_epochs=5,            # allow early stopping to pick best
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    warmup_ratio=0.10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    seed=SEED,
    save_total_limit=2,
    optim="adamw_torch",
    gradient_accumulation_steps=4,  # effective batch size 16
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    lr_scheduler_type="cosine",
    report_to="none",
)

data_collator = DataCollatorWithPadding(tokenizer)

# ==================== Initialize Trainer ====================
trainer = FocalLossWeightedSamplerTrainer(
    pos_weight=pos_weights,
    alpha=FOCAL_ALPHA,
    gamma=FOCAL_GAMMA,
    sample_weights=train_example_weights,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics_multilabel,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# ==================== Train ====================
print("\n" + "="*60)
print("Starting training...")
print("="*60 + "\n")

trainer.train()

# ==================== Evaluate (threshold=0.5, monitoring) ====================
eval_results = trainer.evaluate()
print("\n" + "="*60)
print("Validation Results (threshold=0.5; monitoring only)")
print("="*60)
print(f"Macro F1: {eval_results['eval_f1_macro']:.4f}")
print(f"Micro F1: {eval_results['eval_f1_micro']:.4f}")
print(f"Hamming Loss: {eval_results['eval_hamming_loss']:.4f}")

print("\nPer-label F1 scores:")
for label in LABEL_KEYS:
    print(f"  {label}: {eval_results[f'eval_f1_{label}']:.4f}")

# ==================== Threshold Tuning on Validation ====================
val_pred = trainer.predict(val_dataset)
val_logits = val_pred.predictions
val_true = val[LABEL_KEYS].values.astype(int)

thresholds = tune_thresholds_per_label(val_logits, val_true, LABEL_KEYS, THRESH_GRID)

val_preds_tuned, _ = predict_with_thresholds(val_logits, thresholds)
val_macro_tuned = f1_score(val_true, val_preds_tuned, average="macro", zero_division=0)
val_micro_tuned = f1_score(val_true, val_preds_tuned, average="micro", zero_division=0)
val_hl_tuned = hamming_loss(val_true, val_preds_tuned)

print("\n" + "="*60)
print("Validation Results (tuned thresholds; this is what you want to optimize)")
print("="*60)
print(f"Macro F1: {val_macro_tuned:.4f}")
print(f"Micro F1: {val_micro_tuned:.4f}")
print(f"Hamming Loss: {val_hl_tuned:.4f}")


Positive class pos_weight (N_neg / N_pos) per label (smoothed):
  stereotype         pos=17649  pos_weight=1.972
  vilification       pos=16298  pos_weight=2.219
  dehumanization     pos= 5992  pos_weight=7.754
  extreme_language   pos=11333  pos_weight=3.629
  lack_of_empathy    pos= 9843  pos_weight=4.329
  invalidation       pos= 8625  pos_weight=5.082

Sampler diagnostics:
  Example weights: min=0.178, mean=1.000, max=3.432
  Language counts (train): {'amh': 2830, 'arb': 2856, 'ben': 2860, 'deu': 2714, 'eng': 2725, 'fas': 2797, 'hau': 3108, 'hin': 2353, 'khm': 5679, 'nep': 1691, 'ori': 1999, 'pan': 1433, 'spa': 2798, 'swa': 5953, 'tel': 1992, 'tur': 1988, 'urd': 3045, 'zho': 3640}


/tmp/ipython-input-3473121350.py:82: RuntimeWarning: invalid value encountered in divide
  pos_mean = np.where(pos_count > 0, pos_weight_sum / pos_count, 0.0)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at Sami92/XLM-R-Large-Polarization-Classifier and are newly initialized because the shapes did not match:
- classifier.out_proj.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([6]) in the model instantiated
- classifier.out_proj.weight: found shape torch.Size([2, 1024]) in the checkpoint and torch.Size([6, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



✓ Enabled gradient checkpointing

✓ Using LoRA adapters (PEFT)
trainable params: 3,415,046 || all params: 563,311,628 || trainable%: 0.6062

Starting training...



Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,Hamming Loss,F1 Stereotype,F1 Vilification,F1 Dehumanization,F1 Extreme Language,F1 Lack Of Empathy,F1 Invalidation,Runtime,Samples Per Second,Steps Per Second
1,0.121000,0.100955,0.506304,0.519389,0.248110,0.589897,0.609590,0.387969,0.536522,0.450183,0.463664,58.951700,157.044000,19.643000
2,0.115000,0.091695,0.525249,0.542301,0.212321,0.613246,0.605402,0.418136,0.546864,0.458528,0.509317,58.080000,159.401000,19.938000
3,0.111500,0.094458,0.547280,0.564745,0.231457,0.647022,0.644623,0.439489,0.566645,0.484501,0.501400,57.371900,161.368000,20.184000
4,0.110800,0.091328,0.543963,0.559711,0.223608,0.634719,0.629519,0.434757,0.564191,0.493513,0.507079,56.692600,163.302000,20.426000
5,0.108200,0.091649,0.549802,0.566504,0.223662,0.647892,0.635936,0.437048,0.567660,0.498707,0.511567,58.440100,158.419000,19.815000



Validation Results (threshold=0.5; monitoring only)
Macro F1: 0.5498
Micro F1: 0.5665
Hamming Loss: 0.2237

Per-label F1 scores:
  stereotype: 0.6479
  vilification: 0.6359
  dehumanization: 0.4370
  extreme_language: 0.5677
  lack_of_empathy: 0.4987
  invalidation: 0.5116

Best per-label thresholds (optimized on validation):
  stereotype         thr=0.45  val_f1=0.6695
  vilification       thr=0.45  val_f1=0.6559
  dehumanization     thr=0.55  val_f1=0.4649
  extreme_language   thr=0.47  val_f1=0.5686
  lack_of_empathy    thr=0.50  val_f1=0.4987
  invalidation       thr=0.53  val_f1=0.5232

Validation Results (tuned thresholds; this is what you want to optimize)
Macro F1: 0.5635
Micro F1: 0.5913
Hamming Loss: 0.2195


In [ ]:
# @title
# ==================== Load Development Data ====================
dev_possible_paths = [
    '/content/subtask3/dev',
    '/content/dev',
    './subtask3/dev',
    './dev',
]

dev_base_path = None
for p in dev_possible_paths:
    if os.path.exists(p):
        dev_files = glob.glob(os.path.join(p, "*.csv"))
        if dev_files:
            dev_base_path = p
            break

if dev_base_path is None:
    raise ValueError("❌ Could not find dev data folder. Please verify dev path (e.g., /content/subtask3/dev).")

all_dev_files = glob.glob(os.path.join(dev_base_path, "*.csv"))

dev_df_list = []
for f in all_dev_files:
    lang_code = os.path.basename(f).split('.')[0]
    temp_df = pd.read_csv(f)
    temp_df['lang'] = lang_code
    dev_df_list.append(temp_df)

dev_all = pd.concat(dev_df_list, ignore_index=True)
dev_all = normalize_columns(dev_all)

if "text" not in dev_all.columns or "id" not in dev_all.columns:
    raise ValueError("Dev CSVs must include 'id' and 'text' columns.")

print(f"\nDevelopment set size: {len(dev_all)}")
print(f"Languages: {sorted(dev_all['lang'].unique())}")

# ==================== Generate Predictions (Dev) ====================
pred_dataset = PredictionDataset(dev_all['text'].tolist(), tokenizer, max_length=MAX_LEN)
dev_pred = trainer.predict(pred_dataset)
dev_logits = dev_pred.predictions

# Apply tuned thresholds (critical)
dev_labels_bin, dev_probs = predict_with_thresholds(dev_logits, thresholds)

In [ ]:
# @title
# ==================== Save Predictions ====================
pred_df = dev_all.copy()
for i, label in enumerate(LABEL_KEYS):
    pred_df[label] = dev_labels_bin[:, i].astype(int)

output_dir = 'subtask_3'
os.makedirs(output_dir, exist_ok=True)

languages = sorted(pred_df['lang'].unique())
for lang in languages:
    lang_df = pred_df[pred_df['lang'] == lang].copy()
    output_df = lang_df[['id'] + LABEL_KEYS]
    output_file = os.path.join(output_dir, f'pred_{lang}.csv')
    output_df.to_csv(output_file, index=False)
    print(f"Saved {len(output_df)} predictions for {lang}")

print(f"\n✓ All predictions saved to '{output_dir}/'")

# ==================== Zip + Download (Colab) ====================
try:
    from google.colab import files
    shutil.make_archive("subtask_3", 'zip', "subtask_3")
    files.download("subtask_3.zip")
except Exception as e:
    # Not running in Colab
    shutil.make_archive("subtask_3", 'zip', "subtask_3")
    print("\nCreated subtask_3.zip (not in Colab; skipping auto-download).")
    print("Reason:", str(e))

In [ ]:
# @title
import os
import glob
import shutil
import random
import numpy as np
import pandas as pd
from typing import Any, Dict, List, Union, Optional, Tuple # Import Optional, Tuple

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, hamming_loss

import wandb
wandb.init(mode="disabled")

# ==================== Configuration ====================
LABEL_KEYS = [
    "stereotype",
    "vilification",
    "dehumanization",
    "extreme_language",
    "lack_of_empathy",
    "invalidation"
]

SEED = 42
MAX_LEN = 256

# Focal loss params
FOCAL_ALPHA = 0.25
FOCAL_GAMMA = 2.0

# Threshold search grid
THRESH_GRID = np.linspace(0.05, 0.95, 37)

# Sampling controls
LANG_BALANCE_POWER = 0.5      # softer than 1.0; avoids over-distorting language distribution
LABEL_AWARE_WEIGHT = 0.7      # strength of positive oversampling inside each language
WEIGHT_CLIP_PCT = (5, 95)     # stabilize sampler weights

# Loss scaling controls (language-label rarity)
USE_LANG_LABEL_LOSS_SCALING = True
LOSS_SCALE_POWER = 0.50       # sqrt scaling is typically stable
LOSS_SCALE_CLIP = (0.7, 2.0)  # cap per-language scaling to avoid exploding gradients

# Optional LoRA
USE_LORA_IF_AVAILABLE = True

# ==================== Utilities ====================
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

def normalize_columns(df):
    df.columns = df.columns.str.replace("\n", " ").str.strip()
    df.columns = df.columns.str.replace(r"\s+", " ", regex=True)
    df.columns = df.columns.str.lower().str.replace(" ", "_")
    return df

def sigmoid_np(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))

# ==================== Load and Prepare Data ====================
possible_paths = [
    '/content/subtask3/train/',
    '/content/train/',
    './subtask3/train/',
    './train/',
    '/mnt/data/',
]

base_path = None
for path in possible_paths:
    if os.path.exists(path):
        files = glob.glob(os.path.join(path, "*.csv"))
        if len(files) >= 2:
            base_path = path
            print(f"\u2713 Found training data in: {base_path}")
            print(f"  Files found: {len(files)}")
            break

if base_path is None:
    raise ValueError("Training data not found. Upload data or update base_path.")

all_files = glob.glob(os.path.join(base_path, "*.csv"))

df_list = []
for f in all_files:
    lang_code = os.path.basename(f).split('.')[0]
    temp_df = pd.read_csv(f)
    temp_df['lang'] = lang_code
    df_list.append(temp_df)
    print(f"  Loaded {lang_code}: {len(temp_df)} rows")

df_all = pd.concat(df_list, ignore_index=True)
df_all = normalize_columns(df_all)

if "text" not in df_all.columns:
    raise ValueError("Expected 'text' column in training CSVs but none found.")

for k in LABEL_KEYS:
    if k in df_all.columns:
        df_all[k] = df_all[k].fillna(0).astype(int)
    else:
        df_all[k] = 0

# Language mapping
all_langs = sorted(df_all["lang"].unique().tolist())
lang2id = {l:i for i,l in enumerate(all_langs)}
id2lang = {i:l for l,i in lang2id.items()}

print("\nLanguages:", all_langs)

# ==================== Train/Val Split ====================
train, val = train_test_split(
    df_all,
    test_size=0.15,
    random_state=SEED
)

train = train.reset_index(drop=True)
val = val.reset_index(drop=True)

print(f"\nTraining set size: {len(train)}")
print(f"Validation set size: {len(val)}")

# ==================== Dataset Classes ====================
class PolarizationDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_length=256):
        self.texts = df["text"].astype(str).tolist()
        self.labels = df[LABEL_KEYS].values.astype(np.float32)
        self.lang_ids = df["lang"].map(lang2id).values.astype(np.int64)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding=False,
            max_length=self.max_length,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)
        item["lang_id"] = torch.tensor(self.lang_ids[idx], dtype=torch.long)
        return item

class PredictionDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_length=256):
        self.texts = df["text"].astype(str).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding=False,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {k: v.squeeze(0) for k, v in enc.items()}

# ==================== Custom Data Collator ====================
class CustomDataCollator(DataCollatorWithPadding):
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        batch = super().__call__(features)

        # Ensure labels are float32 for BCEWithLogitsLoss
        if "labels" in batch:
            batch["labels"] = batch["labels"].float()

            # ✅ Only include lang_id when labels exist (train/val)
            if "lang_id" in features[0]:
                batch["lang_id"] = torch.stack([f["lang_id"] for f in features])

        # ✅ If no labels (predict), do NOT pass lang_id to the model
        return batch

# ==================== Global + Per-language pos_weight ====================
def compute_pos_weight(df: pd.DataFrame, eps: float = 1.0) -> torch.Tensor:
    y = df[LABEL_KEYS].astype(float)
    n = len(y)
    pos = y.sum(axis=0).values
    neg = n - pos
    pw = ((neg + eps) / (pos + eps)).astype(np.float32)
    return torch.tensor(pw, dtype=torch.float32)

def compute_lang_label_pos_weight(df: pd.DataFrame, eps: float = 1.0) -> torch.Tensor:
    """
    Returns [num_lang, num_labels] pos_weight_lang[label] = (neg+eps)/(pos+eps) within that language.
    """
    num_lang = len(all_langs)
    num_labels = len(LABEL_KEYS)
    mat = np.zeros((num_lang, num_labels), dtype=np.float32)

    for lang in all_langs:
        sub = df[df["lang"] == lang]
        y = sub[LABEL_KEYS].astype(float)
        n = len(y)
        pos = y.sum(axis=0).values
        neg = n - pos
        mat[lang2id[lang], :] = ((neg + eps) / (pos + eps)).astype(np.float32)

    return torch.tensor(mat, dtype=torch.float32)

global_pos_weight = compute_pos_weight(train, eps=1.0)
lang_label_pos_weight = compute_lang_label_pos_weight(train, eps=1.0)

print("\nGlobal pos_weight (N_neg/N_pos) per label:")
for k, w in zip(LABEL_KEYS, global_pos_weight.tolist()):
    print(f"  {k:18s} {w:.3f}")

# Loss scaling matrix (relative to global)
if USE_LANG_LABEL_LOSS_SCALING:
    # ratio > 1 => rarer in that language than globally => upweight
    ratio = (lang_label_pos_weight / global_pos_weight.unsqueeze(0)).clamp(min=1e-6)
    # sqrt scaling is stable
    loss_scale = ratio.pow(LOSS_SCALE_POWER).clamp(min=LOSS_SCALE_CLIP[0], max=LOSS_SCALE_CLIP[1])
else:
    loss_scale = torch.ones_like(lang_label_pos_weight)

# ==================== Language\u2013label-aware sampler weights ====================
def build_example_weights(df: pd.DataFrame) -> np.ndarray:
    # Language balancing (inverse frequency, softened)
    lang_counts = df["lang"].value_counts().to_dict()
    inv_lang = {l: (1.0 / c) ** LANG_BALANCE_POWER for l, c in lang_counts.items()}

    y = df[LABEL_KEYS].values.astype(np.float32)           # [N, L]
    lang_ids = df["lang"].map(lang2id).values.astype(int)  # [N]

    # Per-example positive rarity within its language
    # pos_weight_lang[lang_id, label] used only where y==1
    pw_lang = lang_label_pos_weight.detach().cpu().numpy()  # [G, L]

    pos_mask = y > 0.5
    # sum of per-language pos_weight over positive labels
    pos_weight_sum = (pos_mask * pw_lang[lang_ids]).sum(axis=1)
    pos_count = pos_mask.sum(axis=1).astype(np.float32)

    pos_mean = np.zeros(len(df), dtype=np.float32)
    nz = pos_count > 0
    pos_mean[nz] = pos_weight_sum[nz] / pos_count[nz]

    label_factor = 1.0 + (LABEL_AWARE_WEIGHT * pos_mean)

    lang_factor = np.array([inv_lang[l] for l in df["lang"].tolist()], dtype=np.float32)

    w = (lang_factor * label_factor).astype(np.float32)

    lo, hi = np.percentile(w, WEIGHT_CLIP_PCT[0]), np.percentile(w, WEIGHT_CLIP_PCT[1])
    w = np.clip(w, lo, hi)
    w = w / (w.mean() + 1e-8)

    print("\nSampler diagnostics:")
    print(f"  weights: min={w.min():.3f}, mean={w.mean():.3f}, max={w.max():.3f}")
    print("  language counts (train):", dict(sorted(lang_counts.items(), key=lambda x: x[0])))

    return w

train_example_weights = build_example_weights(train)

# ==================== Focal Loss (multi-label) ====================
def focal_loss_with_logits(
    logits: torch.Tensor,
    targets: torch.Tensor,
    pos_weight: torch.Tensor,
    gamma: float = 2.0,
    alpha: float = 0.25,
):
    """
    Returns elementwise loss [B, L]
    """
    bce = nn.functional.binary_cross_entropy_with_logits(
        logits, targets, reduction="none", pos_weight=pos_weight.to(logits.device)
    )
    p = torch.sigmoid(logits)
    pt = p * targets + (1.0 - p) * (1.0 - targets)
    alpha_t = alpha * targets + (1.0 - alpha) * (1.0 - targets)
    loss = alpha_t * ((1.0 - pt) ** gamma) * bce
    return loss  # [B, L]

# ==================== Metrics (monitoring at 0.5) ====================
def compute_metrics_multilabel(eval_pred):
    logits, labels = eval_pred
    probs = sigmoid_np(logits)
    preds = (probs >= 0.5).astype(int)
    labels_int = labels.astype(int)

    per_label_f1 = f1_score(labels_int, preds, average=None, zero_division=0)
    metrics = {
        "f1_macro": f1_score(labels_int, preds, average="macro", zero_division=0),
        "f1_micro": f1_score(labels_int, preds, average="micro", zero_division=0),
        "hamming_loss": hamming_loss(labels_int, preds),
    }
    for k, s in zip(LABEL_KEYS, per_label_f1):
        metrics[f"f1_{k}"] = float(s)
    return metrics

# ==================== Threshold tuning ====================
def tune_thresholds_global(val_logits, val_true):
    probs = sigmoid_np(val_logits)
    th = np.zeros(probs.shape[1], dtype=float)
    for j in range(probs.shape[1]):
        best_f1, best_t = -1.0, 0.5
        for t in THRESH_GRID:
            pred = (probs[:, j] >= t).astype(int)
            f1 = f1_score(val_true[:, j], pred, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, float(t)
        th[j] = best_t
    return th

def tune_thresholds_per_language(val_df: pd.DataFrame, val_logits: np.ndarray):
    """
    Returns dict: lang -> thresholds [L].
    If a label has no positives in that language’s val slice, fall back to global threshold.
    """
    global_th = tune_thresholds_global(val_logits, val_df[LABEL_KEYS].values.astype(int))

    probs = sigmoid_np(val_logits)
    out = {}

    for lang in sorted(val_df["lang"].unique().tolist()):
        idx = np.where(val_df["lang"].values == lang)[0]
        y = val_df.iloc[idx][LABEL_KEYS].values.astype(int)
        p = probs[idx]

        th = np.array(global_th, copy=True)

        for j in range(len(LABEL_KEYS)):
            # If language slice has no positives or no negatives, F1 tuning degenerates; keep global
            if y[:, j].sum() == 0 or y[:, j].sum() == len(y):
                continue

            best_f1, best_t = -1.0, th[j]
            for t in THRESH_GRID:
                pred = (p[:, j] >= t).astype(int)
                f1 = f1_score(y[:, j], pred, zero_division=0)
                if f1 > best_f1:
                    best_f1, best_t = f1, float(t)
            th[j] = best_t

        out[lang] = th

    print("\nSample of per-language thresholds (first 5 langs):")
    for lang in list(out.keys())[:5]:
        print(f"  {lang}: {np.round(out[lang], 2)}")

    return out, global_th

def apply_lang_thresholds(df: pd.DataFrame, logits: np.ndarray, th_by_lang: dict, global_th: np.ndarray):
    probs = sigmoid_np(logits)
    preds = np.zeros_like(probs, dtype=int)
    for i, lang in enumerate(df["lang"].tolist()):
        th = th_by_lang.get(lang, global_th)
        preds[i] = (probs[i] >= th).astype(int)
    return preds, probs

# ==================== Custom Trainer (loss scaling + weighted sampler) ====================
class FocalLossLangAwareTrainer(Trainer):
    def __init__(self, pos_weight=None, loss_scale=None, alpha=0.25, gamma=2.0, sample_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight
        self.loss_scale = loss_scale  # [num_lang, num_labels]
        self.alpha = alpha
        self.gamma = gamma
        self.sample_weights = sample_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
      labels = inputs.pop("labels").float()

      # \u2705 robust: lang_id can be missing if something upstream stripped it
      lang_id = inputs.pop("lang_id", None)
      if lang_id is None:
          # fallback: treat everything as lang 0 (won't be used if loss_scale is None)
          lang_id = torch.zeros(labels.size(0), dtype=torch.long, device=labels.device)
      else:
          lang_id = lang_id.to(labels.device)

      outputs = model(**inputs)
      logits = outputs.logits

      loss_elem = focal_loss_with_logits(
          logits=logits,
          targets=labels,
          pos_weight=self.pos_weight,
          gamma=self.gamma,
          alpha=self.alpha
      )

      if self.loss_scale is not None:
          scale = self.loss_scale.to(logits.device)[lang_id]
          loss_elem = loss_elem * scale

      loss = loss_elem.mean()
      return (loss, outputs) if return_outputs else loss

    def get_train_dataloader(self):
        if self.sample_weights is None:
            return super().get_train_dataloader()

        sampler = WeightedRandomSampler(
            weights=torch.tensor(self.sample_weights, dtype=torch.float),
            num_samples=len(self.sample_weights),
            replacement=True
        )
        return DataLoader(
            self.train_dataset,
            batch_size=self.args.train_batch_size,
            sampler=sampler,
            collate_fn=self.data_collator, # Use the custom data collator here
            num_workers=self.args.dataloader_num_workers,
            pin_memory=True
        )

    def prediction_step(
        self,
        model: nn.Module,
        inputs: Dict[str, Union[torch.Tensor, Any]],
        prediction_loss_only: bool,
        ignore_keys: Optional[List[str]] = None
    ) -> Tuple[Optional[torch.Tensor], Optional[torch.Tensor], Optional[torch.Tensor]]:
        # Pop lang_id before passing inputs to the model, as the model's forward
        # method (especially the base XLM-R) does not expect it.
        if "lang_id" in inputs:
            _ = inputs.pop("lang_id")

        return super().prediction_step(
            model, inputs, prediction_loss_only, ignore_keys=ignore_keys
        )

# ==================== Model and Tokenizer ====================
MODEL_NAME = "Sami92/XLM-R-Large-Polarization-Classifier"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_KEYS),
    problem_type="multi_label_classification",
    ignore_mismatched_sizes=True
)

# Gradient checkpointing
try:
    model.gradient_checkpointing_enable()
    print("\n\u2713 Enabled gradient checkpointing")
except Exception as e:
    print("\n(i) Gradient checkpointing not enabled:", str(e))

# Optional LoRA
if USE_LORA_IF_AVAILABLE:
    try:
        from peft import LoraConfig, get_peft_model, TaskType
        lora_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            target_modules=["query", "key", "value"]
        )
        model = get_peft_model(model, lora_config)
        print("\n\u2713 Using LoRA adapters (PEFT)")
        try:
            model.print_trainable_parameters()
        except Exception:
            pass
    except Exception as e:
        print("\n(i) PEFT/LoRA not available; continuing without LoRA.")
        print("    Reason:", str(e))

# ==================== Datasets ====================
train_dataset = PolarizationDataset(train, tokenizer, max_length=MAX_LEN)
val_dataset = PolarizationDataset(val, tokenizer, max_length=MAX_LEN)

# Instantiate the custom data collator
data_collator = CustomDataCollator(tokenizer)

# ==================== Training Arguments ====================
training_args = TrainingArguments(
    output_dir="./checkpoints_xlmr_large_langaware",
    num_train_epochs=7,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    warmup_ratio=0.10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    seed=SEED,
    save_total_limit=2,
    optim="adamw_torch",
    gradient_accumulation_steps=4,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    lr_scheduler_type="cosine",
    report_to="none",
    remove_unused_columns=False,   # \u2705 CRITICAL: keep lang_id in eval/predict
)


# ==================== Trainer ====================
trainer = FocalLossLangAwareTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator, # Pass the custom data collator to the trainer
    compute_metrics=compute_metrics_multilabel,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    pos_weight=global_pos_weight,
    loss_scale=loss_scale,
    alpha=FOCAL_ALPHA,
    gamma=FOCAL_GAMMA,
    sample_weights=train_example_weights
)

# ==================== Train ====================
print("\n" + "="*60)
print("Starting training...")
print("="*60 + "\n")
trainer.train()

# ==================== Evaluate (0.5 monitoring) ====================
eval_results = trainer.evaluate()
print("\n" + "="*60)
print("Validation Results (threshold=0.5; monitoring only)")
print("="*60)
print(f"Macro F1: {eval_results['eval_f1_macro']:.4f}")
print(f"Micro F1: {eval_results['eval_f1_micro']:.4f}")
print(f"Hamming Loss: {eval_results['eval_hamming_loss']:.4f}")

# ==================== Tune thresholds per language ====================
val_pred = trainer.predict(val_dataset)
val_logits = val_pred.predictions

th_by_lang, global_th = tune_thresholds_per_language(val, val_logits)

# Report tuned performance on val using language-specific thresholds
val_preds_lang, _ = apply_lang_thresholds(val, val_logits, th_by_lang, global_th)
val_true = val[LABEL_KEYS].values.astype(int)

val_macro_lang = f1_score(val_true, val_preds_lang, average="macro", zero_division=0)
val_micro_lang = f1_score(val_true, val_preds_lang, average="micro", zero_division=0)
val_hl_lang = hamming_loss(val_true, val_preds_lang)

print("\n" + "="*60)
print("Validation Results (language-specific thresholds)")
print("="*60)
print(f"Macro F1: {val_macro_lang:.4f}")
print(f"Micro F1: {val_micro_lang:.4f}")
print(f"Hamming Loss: {val_hl_lang:.4f}")



/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


✓ Found training data in: /content/subtask3/train/
  Files found: 18
  Loaded khm: 6640 rows
  Loaded pan: 1700 rows
  Loaded deu: 3180 rows
  Loaded spa: 3305 rows
  Loaded hau: 3651 rows
  Loaded nep: 2005 rows
  Loaded tel: 2366 rows
  Loaded zho: 4280 rows
  Loaded arb: 3380 rows
  Loaded urd: 3563 rows
  Loaded hin: 2744 rows
  Loaded tur: 2364 rows
  Loaded ben: 3333 rows
  Loaded ori: 2368 rows
  Loaded eng: 3222 rows
  Loaded swa: 6991 rows
  Loaded fas: 3295 rows
  Loaded amh: 3332 rows

Languages: ['amh', 'arb', 'ben', 'deu', 'eng', 'fas', 'hau', 'hin', 'khm', 'nep', 'ori', 'pan', 'spa', 'swa', 'tel', 'tur', 'urd', 'zho']

Training set size: 52461
Validation set size: 9258

Global pos_weight (N_neg/N_pos) per label:
  stereotype         1.969
  vilification       2.194
  dehumanization     7.643
  extreme_language   3.553
  lack_of_empathy    4.303
  invalidation       5.081

Sampler diagnostics:
  weights: min=0.320, mean=1.000, max=3.399
  language counts (train): {'amh': 2

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at Sami92/XLM-R-Large-Polarization-Classifier and are newly initialized because the shapes did not match:
- classifier.out_proj.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([6]) in the model instantiated
- classifier.out_proj.weight: found shape torch.Size([2, 1024]) in the checkpoint and torch.Size([6, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



✓ Enabled gradient checkpointing

✓ Using LoRA adapters (PEFT)
trainable params: 3,415,046 || all params: 563,311,628 || trainable%: 0.6062

Starting training...



Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,Hamming Loss,F1 Stereotype,F1 Vilification,F1 Dehumanization,F1 Extreme Language,F1 Lack Of Empathy,F1 Invalidation,Runtime,Samples Per Second,Steps Per Second
1,0.149100,0.086960,0.184999,0.225608,0.219612,0.000654,0.429392,0.279050,0.357367,0.031710,0.011819,56.049800,165.174000,20.660000
2,0.136700,0.080115,0.485933,0.495196,0.227965,0.463353,0.587804,0.396552,0.535865,0.445652,0.486369,57.328900,161.489000,20.199000
3,0.127900,0.078730,0.525629,0.536846,0.221538,0.552332,0.620068,0.444519,0.546143,0.487403,0.503308,55.628200,166.427000,20.817000
4,0.124300,0.075532,0.525920,0.535335,0.210107,0.546797,0.609686,0.445087,0.538801,0.494696,0.520455,55.854300,165.753000,20.733000
5,0.125300,0.076522,0.520798,0.527169,0.220566,0.514610,0.606432,0.435312,0.548505,0.495310,0.524615,57.092300,162.158000,20.283000
6,0.123300,0.075093,0.534123,0.543060,0.211097,0.573960,0.603537,0.445578,0.558140,0.496858,0.526665,56.252700,164.579000,20.586000
7,0.124800,0.075234,0.537180,0.547036,0.210989,0.584141,0.608986,0.443634,0.560280,0.496200,0.529839,60.376400,153.338000,19.180000



Validation Results (threshold=0.5; monitoring only)
Macro F1: 0.5372
Micro F1: 0.5470
Hamming Loss: 0.2110

Sample of per-language thresholds (first 5 langs):
  amh: [0.35 0.42 0.55 0.45 0.3  0.42]
  arb: [0.42 0.48 0.6  0.5  0.48 0.45]
  ben: [0.32 0.37 0.5  0.32 0.25 0.35]
  deu: [0.42 0.42 0.52 0.52 0.48 0.45]
  eng: [0.42 0.37 0.55 0.5  0.42 0.5 ]

Validation Results (language-specific thresholds)
Macro F1: 0.5437
Micro F1: 0.5637
Hamming Loss: 0.2706


In [ ]:
import os
import glob
import shutil
import random
import numpy as np
import pandas as pd
from typing import Any, Dict, List, Union, Optional, Tuple # Import Optional, Tuple
import math

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, hamming_loss

import wandb
wandb.init(mode="disabled")

# ==================== Configuration ====================
LABEL_KEYS = [
    "stereotype",
    "vilification",
    "dehumanization",
    "extreme_language",
    "lack_of_empathy",
    "invalidation"
]

SEED = 42
MAX_LEN = 256

# Focal loss params
FOCAL_ALPHA = 0.25
FOCAL_GAMMA = 2.0

# Threshold search grid
THRESH_GRID = np.linspace(0.05, 0.95, 37)

# Sampling controls
LANG_BALANCE_POWER = 0.5      # softer than 1.0; avoids over-distorting language distribution
LABEL_AWARE_WEIGHT = 0.7      # strength of positive oversampling inside each language
WEIGHT_CLIP_PCT = (5, 95)     # stabilize sampler weights

# Loss scaling controls (language-label rarity)
USE_LANG_LABEL_LOSS_SCALING = True
LOSS_SCALE_POWER = 0.50       # sqrt scaling is typically stable
LOSS_SCALE_CLIP = (0.7, 2.0)  # cap per-language scaling to avoid exploding gradients

# Optional LoRA
USE_LORA_IF_AVAILABLE = True

# ==================== Utilities ====================
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

def normalize_columns(df):
    df.columns = df.columns.str.replace("\n", " ").str.strip()
    df.columns = df.columns.str.replace(r"\s+", " ", regex=True)
    df.columns = df.columns.str.lower().str.replace(" ", "_")
    return df

def sigmoid_np(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))

# ==================== Load and Prepare Data ====================
possible_paths = [
    '/content/subtask3/train/',
    '/content/train/',
    './subtask3/train/',
    './train/',
    '/mnt/data/',
]

base_path = None
for path in possible_paths:
    if os.path.exists(path):
        files = glob.glob(os.path.join(path, "*.csv"))
        if len(files) >= 2:
            base_path = path
            print(f"\u2713 Found training data in: {base_path}")
            print(f"  Files found: {len(files)}")
            break

if base_path is None:
    raise ValueError("Training data not found. Upload data or update base_path.")

all_files = glob.glob(os.path.join(base_path, "*.csv"))

df_list = []
for f in all_files:
    lang_code = os.path.basename(f).split('.')[0]
    temp_df = pd.read_csv(f)
    temp_df['lang'] = lang_code
    df_list.append(temp_df)
    print(f"  Loaded {lang_code}: {len(temp_df)} rows")

df_all = pd.concat(df_list, ignore_index=True)
df_all = normalize_columns(df_all)

if "text" not in df_all.columns:
    raise ValueError("Expected 'text' column in training CSVs but none found.")

for k in LABEL_KEYS:
    if k in df_all.columns:
        df_all[k] = df_all[k].fillna(0).astype(int)
    else:
        df_all[k] = 0

# Language mapping
all_langs = sorted(df_all["lang"].unique().tolist())
lang2id = {l:i for i,l in enumerate(all_langs)}
id2lang = {i:l for l,i in lang2id.items()}

print("\nLanguages:", all_langs)

# ==================== Train/Val Split ====================
def split_by_language(df, val_ratio=0.15, seed=42):
    parts_train, parts_val = [], []
    rng = np.random.RandomState(seed)
    for lang, sub in df.groupby("lang"):
        idx = np.arange(len(sub))
        rng.shuffle(idx)
        n_val = max(1, int(len(sub) * val_ratio))
        val_idx = idx[:n_val]
        tr_idx  = idx[n_val:]
        parts_val.append(sub.iloc[val_idx])
        parts_train.append(sub.iloc[tr_idx])
    return pd.concat(parts_train).reset_index(drop=True), pd.concat(parts_val).reset_index(drop=True)

train, val = split_by_language(df_all, val_ratio=0.15, seed=SEED)

print(f"\nTraining set size: {len(train)}")
print(f"Validation set size: {len(val)}")

# ==================== Dataset Classes ====================
class PolarizationDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_length=256):
        self.texts = df["text"].astype(str).tolist()
        self.labels = df[LABEL_KEYS].values.astype(np.float32)
        self.lang_ids = df["lang"].map(lang2id).values.astype(np.int64)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding=False,
            max_length=self.max_length,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)
        item["lang_id"] = torch.tensor(self.lang_ids[idx], dtype=torch.long)
        return item

class PredictionDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_length=256):
        self.texts = df["text"].astype(str).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding=False,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {k: v.squeeze(0) for k, v in enc.items()}

# ==================== Custom Data Collator ====================
class CustomDataCollator(DataCollatorWithPadding):
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        batch = super().__call__(features)

        # Ensure labels are float32 for BCEWithLogitsLoss
        if "labels" in batch:
            batch["labels"] = batch["labels"].float()

            # ✅ Only include lang_id when labels exist (train/val)
            if "lang_id" in features[0]:
                batch["lang_id"] = torch.stack([f["lang_id"] for f in features])

        # ✅ If no labels (predict), do NOT pass lang_id to the model
        return batch

# ==================== Global + Per-language pos_weight ====================
def compute_pos_weight(df: pd.DataFrame, eps: float = 1.0) -> torch.Tensor:
    y = df[LABEL_KEYS].astype(float)
    n = len(y)
    pos = y.sum(axis=0).values
    neg = n - pos
    pw = ((neg + eps) / (pos + eps)).astype(np.float32)
    return torch.tensor(pw, dtype=torch.float32)

def compute_lang_label_pos_weight(df: pd.DataFrame, eps: float = 1.0) -> torch.Tensor:
    """
    Returns [num_lang, num_labels] pos_weight_lang[label] = (neg+eps)/(pos+eps) within that language.
    """
    num_lang = len(all_langs)
    num_labels = len(LABEL_KEYS)
    mat = np.zeros((num_lang, num_labels), dtype=np.float32)

    for lang in all_langs:
        sub = df[df["lang"] == lang]
        y = sub[LABEL_KEYS].astype(float)
        n = len(y)
        pos = y.sum(axis=0).values
        neg = n - pos
        mat[lang2id[lang], :] = ((neg + eps) / (pos + eps)).astype(np.float32)

    return torch.tensor(mat, dtype=torch.float32)

global_pos_weight = compute_pos_weight(train, eps=1.0)
lang_label_pos_weight = compute_lang_label_pos_weight(train, eps=1.0)

print("\nGlobal pos_weight (N_neg/N_pos) per label:")
for k, w in zip(LABEL_KEYS, global_pos_weight.tolist()):
    print(f"  {k:18s} {w:.3f}")

# Loss scaling matrix (relative to global)
if USE_LANG_LABEL_LOSS_SCALING:
    # ratio > 1 => rarer in that language than globally => upweight
    ratio = (lang_label_pos_weight / global_pos_weight.unsqueeze(0)).clamp(min=1e-6)
    # sqrt scaling is stable
    loss_scale = ratio.pow(LOSS_SCALE_POWER).clamp(min=LOSS_SCALE_CLIP[0], max=LOSS_SCALE_CLIP[1])
else:
    loss_scale = torch.ones_like(lang_label_pos_weight)

# ==================== Language\u2013label-aware sampler weights ====================
def build_example_weights(df: pd.DataFrame) -> np.ndarray:
    # Language balancing (inverse frequency, softened)
    lang_counts = df["lang"].value_counts().to_dict()
    inv_lang = {l: (1.0 / c) ** LANG_BALANCE_POWER for l, c in lang_counts.items()}

    y = df[LABEL_KEYS].values.astype(np.float32)           # [N, L]
    lang_ids = df["lang"].map(lang2id).values.astype(int)  # [N]

    # Per-example positive rarity within its language
    # pos_weight_lang[lang_id, label] used only where y==1
    pw_lang = lang_label_pos_weight.detach().cpu().numpy()  # [G, L]

    pos_mask = y > 0.5
    # sum of per-language pos_weight over positive labels
    pos_weight_sum = (pos_mask * pw_lang[lang_ids]).sum(axis=1)
    pos_count = pos_mask.sum(axis=1).astype(np.float32)

    pos_mean = np.zeros(len(df), dtype=np.float32)
    nz = pos_count > 0
    pos_mean[nz] = pos_weight_sum[nz] / pos_count[nz]

    label_factor = 1.0 + (LABEL_AWARE_WEIGHT * pos_mean)

    lang_factor = np.array([inv_lang[l] for l in df["lang"].tolist()], dtype=np.float32)

    w = (lang_factor * label_factor).astype(np.float32)

    lo, hi = np.percentile(w, WEIGHT_CLIP_PCT[0]), np.percentile(w, WEIGHT_CLIP_PCT[1])
    w = np.clip(w, lo, hi)
    w = w / (w.mean() + 1e-8)

    print("\nSampler diagnostics:")
    print(f"  weights: min={w.min():.3f}, mean={w.mean():.3f}, max={w.max():.3f}")
    print("  language counts (train):", dict(sorted(lang_counts.items(), key=lambda x: x[0])))

    return w

train_example_weights = build_example_weights(train)

def asymmetric_loss_with_logits(
    logits: torch.Tensor,
    targets: torch.Tensor,
    pos_weight: torch.Tensor,
    gamma_neg: float = 4.0,
    gamma_pos: float = 1.0,
    clip: float = 0.05,
):
    """
    ASL for multi-label classification (logits).
    - gamma_neg larger -> suppress easy negatives
    - gamma_pos smaller -> don’t over-penalize positives
    - clip -> stabilizes by clipping negative probs
    Returns elementwise loss [B, L]
    """
    x_sigmoid = torch.sigmoid(logits)
    xs_pos = x_sigmoid
    xs_neg = 1.0 - x_sigmoid

    if clip is not None and clip > 0:
        xs_neg = (xs_neg + clip).clamp(max=1.0)

    # BCE with pos_weight
    bce = nn.functional.binary_cross_entropy_with_logits(
        logits, targets, reduction="none", pos_weight=pos_weight.to(logits.device)
    )

    # focal-like asymmetric focusing
    pt = xs_pos * targets + xs_neg * (1.0 - targets)
    gamma = gamma_pos * targets + gamma_neg * (1.0 - targets)
    weight = (1.0 - pt).pow(gamma)

    return weight * bce

# ==================== Metrics (monitoring at 0.5) ====================
def compute_metrics_multilabel(eval_pred):
    logits, labels = eval_pred
    probs = sigmoid_np(logits)
    preds = (probs >= 0.5).astype(int)
    labels_int = labels.astype(int)

    per_label_f1 = f1_score(labels_int, preds, average=None, zero_division=0)
    metrics = {
        "f1_macro": f1_score(labels_int, preds, average="macro", zero_division=0),
        "f1_micro": f1_score(labels_int, preds, average="micro", zero_division=0),
        "hamming_loss": hamming_loss(labels_int, preds),
    }
    for k, s in zip(LABEL_KEYS, per_label_f1):
        metrics[f"f1_{k}"] = float(s)
    return metrics

# ==================== Threshold tuning ====================
def tune_thresholds_global(val_logits, val_true):
    probs = sigmoid_np(val_logits)
    th = np.zeros(probs.shape[1], dtype=float)
    for j in range(probs.shape[1]):
        best_f1, best_t = -1.0, 0.5
        for t in THRESH_GRID:
            pred = (probs[:, j] >= t).astype(int)
            f1 = f1_score(val_true[:, j], pred, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, float(t)
        th[j] = best_t
    return th

def tune_thresholds_per_language(val_df: pd.DataFrame, val_logits: np.ndarray):
    """
    Returns dict: lang -> thresholds [L].
    If a label has no positives in that language’s val slice, fall back to global threshold.
    """
    global_th = tune_thresholds_global(val_logits, val_df[LABEL_KEYS].values.astype(int))

    probs = sigmoid_np(val_logits)
    out = {}

    for lang in sorted(val_df["lang"].unique().tolist()):
        idx = np.where(val_df["lang"].values == lang)[0]
        y = val_df.iloc[idx][LABEL_KEYS].values.astype(int)
        p = probs[idx]

        th = np.array(global_th, copy=True)

        for j in range(len(LABEL_KEYS)):
            # If language slice has no positives or no negatives, F1 tuning degenerates; keep global
            if y[:, j].sum() == 0 or y[:, j].sum() == len(y):
                continue

            best_f1, best_t = -1.0, th[j]
            for t in THRESH_GRID:
                pred = (p[:, j] >= t).astype(int)
                f1 = f1_score(y[:, j], pred, zero_division=0)
                if f1 > best_f1:
                    best_f1, best_t = f1, float(t)
            th[j] = best_t

        out[lang] = th

    print("\nSample of per-language thresholds (first 5 langs):")
    for lang in list(out.keys())[:5]:
        print(f"  {lang}: {np.round(out[lang], 2)}")

    return out, global_th

def apply_lang_thresholds(df: pd.DataFrame, logits: np.ndarray, th_by_lang: dict, global_th: np.ndarray):
    probs = sigmoid_np(logits)
    preds = np.zeros_like(probs, dtype=int)
    for i, lang in enumerate(df["lang"].tolist()):
        th = th_by_lang.get(lang, global_th)
        preds[i] = (probs[i] >= th).astype(int)
    return preds, probs

# ==================== Calibration: Per-language Temperature Scaling ====================
@torch.no_grad()
def _bce_eval_loss(logits_t: torch.Tensor, labels_t: torch.Tensor, pos_weight: torch.Tensor) -> float:
    # logits_t/labels_t: [N, L]
    loss = nn.functional.binary_cross_entropy_with_logits(
        logits_t, labels_t, reduction="mean", pos_weight=pos_weight.to(logits_t.device)
    )
    return float(loss.item())

def fit_temperature_per_language(
    val_df: pd.DataFrame,
    val_logits: np.ndarray,
    label_keys: List[str],
    lang2id: Dict[str, int],
    pos_weight: torch.Tensor,
    device: Optional[str] = None,
    max_iter: int = 200,
    lr: float = 0.05,
    t_min: float = 0.5,
    t_max: float = 5.0,
    min_samples: int = 200,   # if a language has too few samples, fallback to global T
) -> Dict[str, float]:
    """
    Fits a scalar temperature T per language by minimizing BCEWithLogitsLoss
    on that language's validation examples.

    Returns: dict {lang: T_lang}
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    # tensors once
    logits_all = torch.tensor(val_logits, dtype=torch.float32, device=device)
    y_all = torch.tensor(val_df[label_keys].values.astype(np.float32), device=device)

    langs = sorted(val_df["lang"].unique().tolist())

    # First fit a GLOBAL temperature (fallback/default)
    global_log_t = torch.zeros(1, device=device, requires_grad=True)  # log(T)
    opt = torch.optim.LBFGS([global_log_t], lr=lr, max_iter=max_iter)

    def closure_global():
        opt.zero_grad()
        T = torch.exp(global_log_t).clamp(min=t_min, max=t_max)
        loss = nn.functional.binary_cross_entropy_with_logits(
            logits_all / T, y_all, reduction="mean", pos_weight=pos_weight.to(device)
        )
        loss.backward()
        return loss

    opt.step(closure_global)
    T_global = float(torch.exp(global_log_t).clamp(min=t_min, max=t_max).item())

    temps = {lang: T_global for lang in langs}

    # Then fit per-language temperatures
    for lang in langs:
        idx = np.where(val_df["lang"].values == lang)[0]
        if len(idx) < min_samples:
            temps[lang] = T_global
            continue

        logits_lang = logits_all[idx]
        y_lang = y_all[idx]

        log_t = torch.tensor([math.log(T_global)], device=device, requires_grad=True)
        opt_lang = torch.optim.LBFGS([log_t], lr=lr, max_iter=max_iter)

        def closure_lang():
            opt_lang.zero_grad()
            T = torch.exp(log_t).clamp(min=t_min, max=t_max)
            loss = nn.functional.binary_cross_entropy_with_logits(
                logits_lang / T, y_lang, reduction="mean", pos_weight=pos_weight.to(device)
            )
            loss.backward()
            return loss

        opt_lang.step(closure_lang)
        T_lang = float(torch.exp(log_t).clamp(min=t_min, max=t_max).item())
        temps[lang] = T_lang

    print("\nTemperature scaling (T per language):")
    print(f"  Global T = {T_global:.3f}")
    # show a few
    for lang in list(temps.keys())[:8]:
        print(f"  {lang}: {temps[lang]:.3f}")

    return temps

def apply_temperature_per_language(
    df: pd.DataFrame,
    logits: np.ndarray,
    temps_by_lang: Dict[str, float],
    default_T: float = 1.0
) -> np.ndarray:
    """
    Applies calibrated logits = logits / T_lang row-wise according to df['lang'].
    """
    out = logits.astype(np.float32).copy()
    langs = df["lang"].tolist()
    for i, lang in enumerate(langs):
        T = temps_by_lang.get(lang, default_T)
        if T <= 0:
            T = default_T
        out[i] = out[i] / float(T)
    return out

def evaluate_macro_f1_from_logits_with_lang_thresholds(
    df: pd.DataFrame,
    logits: np.ndarray,
    th_by_lang: Dict[str, np.ndarray],
    global_th: np.ndarray,
    label_keys: List[str]
) -> float:
    preds, _ = apply_lang_thresholds(df, logits, th_by_lang, global_th)
    y_true = df[label_keys].values.astype(int)
    return float(f1_score(y_true, preds, average="macro", zero_division=0))

# ==================== Custom Trainer (loss scaling + weighted sampler) ====================
class FocalLossLangAwareTrainer(Trainer):
    def __init__(self, pos_weight=None, loss_scale=None, alpha=0.25, gamma=2.0, sample_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight
        self.loss_scale = loss_scale  # [num_lang, num_labels]
        self.alpha = alpha
        self.gamma = gamma
        self.sample_weights = sample_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
      labels = inputs.pop("labels").float()

      # \u2705 robust: lang_id can be missing if something upstream stripped it
      lang_id = inputs.pop("lang_id", None)
      if lang_id is None:
          # fallback: treat everything as lang 0 (won't be used if loss_scale is None)
          lang_id = torch.zeros(labels.size(0), dtype=torch.long, device=labels.device)
      else:
          lang_id = lang_id.to(labels.device)

      outputs = model(**inputs)
      logits = outputs.logits

      loss_elem = asymmetric_loss_with_logits(
            logits=logits,
            targets=labels,
            pos_weight=self.pos_weight,
            gamma_neg=4.0,
            gamma_pos=1.0,
            clip=0.05
        )


      if self.loss_scale is not None:
          scale = self.loss_scale.to(logits.device)[lang_id]
          loss_elem = loss_elem * scale

      loss = loss_elem.mean()
      return (loss, outputs) if return_outputs else loss

    def get_train_dataloader(self):
        if self.sample_weights is None:
            return super().get_train_dataloader()

        sampler = WeightedRandomSampler(
            weights=torch.tensor(self.sample_weights, dtype=torch.float),
            num_samples=len(self.sample_weights),
            replacement=True
        )
        return DataLoader(
            self.train_dataset,
            batch_size=self.args.train_batch_size,
            sampler=sampler,
            collate_fn=self.data_collator, # Use the custom data collator here
            num_workers=self.args.dataloader_num_workers,
            pin_memory=True
        )

    def prediction_step(
        self,
        model: nn.Module,
        inputs: Dict[str, Union[torch.Tensor, Any]],
        prediction_loss_only: bool,
        ignore_keys: Optional[List[str]] = None
    ) -> Tuple[Optional[torch.Tensor], Optional[torch.Tensor], Optional[torch.Tensor]]:
        # Pop lang_id before passing inputs to the model, as the model's forward
        # method (especially the base XLM-R) does not expect it.
        if "lang_id" in inputs:
            _ = inputs.pop("lang_id")

        return super().prediction_step(
            model, inputs, prediction_loss_only, ignore_keys=ignore_keys
        )

# ==================== Model and Tokenizer ====================
MODEL_NAME = "microsoft/mdeberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_KEYS),
    problem_type="multi_label_classification",
    ignore_mismatched_sizes=True
)

# Gradient checkpointing
try:
    # model.gradient_checkpointing_enable() # Commented out to fix the RuntimeError
    print("\n(i) Gradient checkpointing explicitly disabled to avoid RuntimeError.")
except Exception as e:
    print("\n(i) Gradient checkpointing not enabled:", str(e))

# Optional LoRA
if USE_LORA_IF_AVAILABLE:
    try:
        from peft import LoraConfig, get_peft_model, TaskType
        lora_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            target_modules=["query", "key", "value"]
        )
        model = get_peft_model(model, lora_config)
        print("\n\u2713 Using LoRA adapters (PEFT)")
        try:
            model.print_trainable_parameters()
        except Exception:
            pass
    except Exception as e:
        print("\n(i) PEFT/LoRA not available; continuing without LoRA.")
        print("    Reason:", str(e))

# ==================== Datasets ====================
train_dataset = PolarizationDataset(train, tokenizer, max_length=MAX_LEN)
val_dataset = PolarizationDataset(val, tokenizer, max_length=MAX_LEN)

# Instantiate the custom data collator
data_collator = CustomDataCollator(tokenizer)

# ==================== Training Arguments ====================
training_args = TrainingArguments(
    output_dir="./checkpoints_xlmr_large_langaware",
    num_train_epochs=6,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    warmup_ratio=0.10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    seed=SEED,
    save_total_limit=2,
    optim="adamw_torch",
    gradient_accumulation_steps=4,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    lr_scheduler_type="cosine",
    report_to="none",
    remove_unused_columns=False,   # \u2705 CRITICAL: keep lang_id in eval/predict
)


# ==================== Trainer ====================
trainer = FocalLossLangAwareTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator, # Pass the custom data collator to the trainer
    compute_metrics=compute_metrics_multilabel,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    pos_weight=global_pos_weight,
    loss_scale=loss_scale,
    alpha=FOCAL_ALPHA,
    gamma=FOCAL_GAMMA,
    sample_weights=train_example_weights
)

# ==================== Train ====================
print("\n" + "="*60)
print("Starting training...")
print("="*60 + "\n")
trainer.train()

# ==================== Evaluate (0.5 monitoring) ====================
eval_results = trainer.evaluate()
print("\n" + "="*60)
print("Validation Results (threshold=0.5; monitoring only)")
print("="*60)
print(f"Macro F1: {eval_results['eval_f1_macro']:.4f}")
print(f"Micro F1: {eval_results['eval_f1_micro']:.4f}")
print(f"Hamming Loss: {eval_results['eval_hamming_loss']:.4f}")

# ==================== Tune thresholds per language ====================
val_pred = trainer.predict(val_dataset)
val_logits = val_pred.predictions

# ==================== Per-language temperature scaling (fit on val) ====================
temps_by_lang = fit_temperature_per_language(
    val_df=val,
    val_logits=val_logits,
    label_keys=LABEL_KEYS,
    lang2id=lang2id,
    pos_weight=global_pos_weight,
    min_samples=200
)

# Apply calibration to validation logits
val_logits_cal = apply_temperature_per_language(val, val_logits, temps_by_lang, default_T=1.0)

th_by_lang, global_th = tune_thresholds_per_language(val, val_logits_cal)

# Report tuned performance on val using language-specific thresholds
val_preds_lang, _ = apply_lang_thresholds(val, val_logits_cal, th_by_lang, global_th)
val_true = val[LABEL_KEYS].values.astype(int)

val_macro_lang = f1_score(val_true, val_preds_lang, average="macro", zero_division=0)
val_micro_lang = f1_score(val_true, val_preds_lang, average="micro", zero_division=0)
val_hl_lang = hamming_loss(val_true, val_preds_lang)

macro_before = evaluate_macro_f1_from_logits_with_lang_thresholds(val, val_logits, th_by_lang, global_th, LABEL_KEYS)
macro_after  = evaluate_macro_f1_from_logits_with_lang_thresholds(val, val_logits_cal, th_by_lang, global_th, LABEL_KEYS)
print(f"\nVal Macro-F1 (uncalibrated logits): {macro_before:.4f}")
print(f"Val Macro-F1 (temp-scaled logits):   {macro_after:.4f}")

print("\n" + "="*60)
print("Validation Results (language-specific thresholds)")
print("="*60)
print(f"Macro F1: {val_macro_lang:.4f}")
print(f"Micro F1: {val_micro_lang:.4f}")
print(f"Hamming Loss: {val_hl_lang:.4f}")

✓ Found training data in: /content/subtask3/train/
  Files found: 18
  Loaded urd: 3563 rows
  Loaded nep: 2005 rows
  Loaded ori: 2368 rows
  Loaded arb: 3380 rows
  Loaded fas: 3295 rows
  Loaded hin: 2744 rows
  Loaded deu: 3180 rows
  Loaded swa: 6991 rows
  Loaded khm: 6640 rows
  Loaded pan: 1700 rows
  Loaded hau: 3651 rows
  Loaded spa: 3305 rows
  Loaded tel: 2366 rows
  Loaded amh: 3332 rows
  Loaded eng: 3222 rows
  Loaded tur: 2364 rows
  Loaded zho: 4280 rows
  Loaded ben: 3333 rows

Languages: ['amh', 'arb', 'ben', 'deu', 'eng', 'fas', 'hau', 'hin', 'khm', 'nep', 'ori', 'pan', 'spa', 'swa', 'tel', 'tur', 'urd', 'zho']

Training set size: 52469
Validation set size: 9250

Global pos_weight (N_neg/N_pos) per label:
  stereotype         1.974
  vilification       2.210
  dehumanization     7.641
  extreme_language   3.555
  lack_of_empathy    4.296
  invalidation       5.079

Sampler diagnostics:
  weights: min=0.321, mean=1.000, max=3.377
  language counts (train): {'amh': 2

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/mdeberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



(i) Gradient checkpointing explicitly disabled to avoid RuntimeError.

(i) PEFT/LoRA not available; continuing without LoRA.
    Reason: No modules were targeted for adaptation. This might be caused by a combination of mismatched target modules and excluded modules. Please check your `target_modules` and `exclude_modules` configuration. You may also have only targeted modules that are marked to be saved (`modules_to_save`).

Starting training...



Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,Hamming Loss,F1 Stereotype,F1 Vilification,F1 Dehumanization,F1 Extreme Language,F1 Lack Of Empathy,F1 Invalidation,Runtime,Samples Per Second,Steps Per Second
1,0.236700,0.159758,0.420634,0.431743,0.563946,0.544428,0.554894,0.317730,0.431675,0.333201,0.341876,40.651300,227.545000,28.462000
2,0.180300,0.169570,0.489912,0.505480,0.403207,0.591378,0.620846,0.397419,0.487434,0.442396,0.400000,40.364500,229.162000,28.664000
3,0.144900,0.177240,0.505572,0.522900,0.373514,0.581804,0.614839,0.415185,0.502352,0.447624,0.471629,40.861700,226.373000,28.315000
4,0.115600,0.205277,0.542027,0.562116,0.303063,0.637076,0.647768,0.446943,0.531330,0.485294,0.503748,41.867000,220.938000,27.635000
5,0.100700,0.224142,0.547873,0.568382,0.291766,0.648789,0.652217,0.452909,0.538497,0.494039,0.500790,40.241100,229.864000,28.752000
6,0.094400,0.228464,0.545605,0.566220,0.294541,0.646620,0.648792,0.450120,0.538121,0.491440,0.498537,40.859600,226.385000,28.316000



Validation Results (threshold=0.5; monitoring only)
Macro F1: 0.5479
Micro F1: 0.5684
Hamming Loss: 0.2918

Temperature scaling (T per language):
  Global T = 1.049
  amh: 1.179
  arb: 1.028
  ben: 1.011
  deu: 1.611
  eng: 1.289
  fas: 1.361
  hau: 0.705
  hin: 0.797

Sample of per-language thresholds (first 5 langs):
  amh: [0.62 0.57 0.55 0.6  0.6  0.72]
  arb: [0.68 0.78 0.72 0.57 0.75 0.45]
  ben: [0.35 0.68 0.37 0.72 0.25 0.57]
  deu: [0.55 0.52 0.6  0.55 0.6  0.48]
  eng: [0.62 0.7  0.5  0.62 0.65 0.52]

Val Macro-F1 (uncalibrated logits): 0.5619
Val Macro-F1 (temp-scaled logits):   0.5787

Validation Results (language-specific thresholds)
Macro F1: 0.5787
Micro F1: 0.6013
Hamming Loss: 0.2266


In [ ]:
from google.colab import drive
import os
import json
import torch

drive.mount('/content/drive')

# Shared parent directory
BASE_DIR = "/content/drive/MyDrive/training"
os.makedirs(BASE_DIR, exist_ok=True)

# Model-specific subfolder
save_path = os.path.join(BASE_DIR, "polarization_modelC_mdeberta")
os.makedirs(save_path, exist_ok=True)

# Save the model weights (or LoRA adapters)
model.save_pretrained(save_path)

# Save the tokenizer
tokenizer.save_pretrained(save_path)

# Convert numpy arrays to JSON-serializable lists
serializable_thresholds = {
    lang: th.tolist() if isinstance(th, np.ndarray) else th
    for lang, th in th_by_lang.items()
}

metadata = {
    "label_keys": LABEL_KEYS,
    "global_thresholds": global_th.tolist(),
    "per_language_thresholds": serializable_thresholds,
    "per_language_temperatures": temps_by_lang,
    "model_name": MODEL_NAME
}

with open(os.path.join(save_path, "post_processing_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=4)

print(f"Model and metadata saved successfully to {save_path}")

Mounted at /content/drive
Model and metadata saved successfully to /content/drive/MyDrive/training/polarization_modelC_mdeberta


In [ ]:

# ==================== Load Dev Data ====================
dev_possible_paths = [
    '/content/subtask3/dev',
    '/content/dev',
    './subtask3/dev',
    './dev',
]
dev_base_path = None
for p in dev_possible_paths:
    if os.path.exists(p) and glob.glob(os.path.join(p, "*.csv")):
        dev_base_path = p
        break
if dev_base_path is None:
    raise ValueError("Dev data folder not found. Verify dev path (e.g., /content/subtask3/dev).")

all_dev_files = glob.glob(os.path.join(dev_base_path, "*.csv"))
dev_df_list = []
for f in all_dev_files:
    lang_code = os.path.basename(f).split('.')[0]
    temp_df = pd.read_csv(f)
    temp_df['lang'] = lang_code
    dev_df_list.append(temp_df)

dev_all = normalize_columns(pd.concat(dev_df_list, ignore_index=True))
if "text" not in dev_all.columns or "id" not in dev_all.columns:
    raise ValueError("Dev CSVs must include 'id' and 'text' columns.")

# Ensure lang codes exist in mapping (should, but guard anyway)
missing_langs = sorted(set(dev_all["lang"].unique()) - set(all_langs))
if missing_langs:
    raise ValueError(f"Dev has languages not seen in train mapping: {missing_langs}")

print(f"\nDevelopment set size: {len(dev_all)}")
print(f"Languages: {sorted(dev_all['lang'].unique())}")

# ==================== Predict on Dev (language-specific thresholds) ====================
pred_dataset = PredictionDataset(dev_all, tokenizer, max_length=MAX_LEN)
dev_pred = trainer.predict(pred_dataset)
dev_logits = dev_pred.predictions

# Apply per-language temperature scaling to dev logits
dev_logits_cal = apply_temperature_per_language(dev_all, dev_logits, temps_by_lang, default_T=1.0)

dev_labels_bin, _ = apply_lang_thresholds(dev_all, dev_logits_cal, th_by_lang, global_th)

# ==================== Save Predictions ====================
pred_df = dev_all.copy()
for j, label in enumerate(LABEL_KEYS):
    pred_df[label] = dev_labels_bin[:, j].astype(int)

output_dir = "subtask_3"
os.makedirs(output_dir, exist_ok=True)

for lang in sorted(pred_df["lang"].unique()):
    out = pred_df[pred_df["lang"] == lang][["id"] + LABEL_KEYS]
    out.to_csv(os.path.join(output_dir, f"pred_{lang}.csv"), index=False)
    print(f"Saved {len(out)} predictions for {lang}")

print(f"\n\u2713 All predictions saved to '{output_dir}/'")

# ==================== Zip + Download ====================
try:
    from google.colab import files
    shutil.make_archive("subtask_3", "zip", output_dir)
    files.download("subtask_3.zip")
except Exception as e:
    shutil.make_archive("subtask_3", "zip", output_dir)
    print("\nCreated subtask_3.zip (not in Colab; skipping auto-download).")
    print("Reason:", str(e))



Development set size: 3091
Languages: ['amh', 'arb', 'ben', 'deu', 'eng', 'fas', 'hau', 'hin', 'khm', 'nep', 'ori', 'pan', 'spa', 'swa', 'tel', 'tur', 'urd', 'zho']


Saved 166 predictions for amh
Saved 169 predictions for arb
Saved 166 predictions for ben
Saved 159 predictions for deu
Saved 160 predictions for eng
Saved 164 predictions for fas
Saved 182 predictions for hau
Saved 137 predictions for hin
Saved 332 predictions for khm
Saved 100 predictions for nep
Saved 118 predictions for ori
Saved 100 predictions for pan
Saved 165 predictions for spa
Saved 349 predictions for swa
Saved 118 predictions for tel
Saved 115 predictions for tur
Saved 177 predictions for urd
Saved 214 predictions for zho

✓ All predictions saved to 'subtask_3/'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>